## Exampling using RAG for research

In [1]:
from dotenv import dotenv_values
import os
import sys
sys.path.insert(0, '/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/arrow/24.0.0/lib/python3.12/site-packages')
sys.path.insert(0, '/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/CUDA/gcc12/cuda12.6/faiss/1.12.0/lib/python3.12/site-packages')

In [2]:
config = dotenv_values(".env")
os.environ["NVIDIA_API_KEY"] = config['NVIDIA_API_KEY']
os.environ["HUGGINGFACEHUB_API_TOKEN"] = config['HF_API_KEY']

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
# ---- LLM: free model with reliable tool calling ----
from crewai import Agent, Task, Crew, LLM

In [5]:
# llm = LLM(
#     model="deepseek/deepseek-v4-flash:free",  #  free, strong tool calling
#     api_key=os.environ["NVIDIA_API_KEY"],         # reuse your OpenRouter key here
#     base_url="https://openrouter.ai/api/v1",
#     # max_rpm=10,        
#     max_tokens=50,
# )

# llm = LLM(
#     model="groq/llama-3.1-8b-instant",
#     api_key=config['GROQ_API_KEY'],  # add to your .env
#     max_tokens=500,
# )

llm = LLM(
    model="openrouter/nvidia/nemotron-3-nano-30b-a3b:free",  
    api_key=config['NVIDIA_API_KEY'],
    base_url="https://openrouter.ai/api/v1",
)

In [6]:
# Alternatives if the above hits rate limits:
# model="meta-llama/llama-4-maverick:free"
# model="qwen/qwen3-235b-a22b:free"

In [7]:
# ---- Build vector DB ----
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

In [8]:
# pdf_mdtb = 'data/mdtb.pdf'
# pdf_hcp = 'data/hcp.pdf'

pdf_1 = 'data/2023_multitask.pdf'
pdf_2 = 'data/2024_demand.pdf'
pdf_3 = 'data/2007_badre.pdf'

docs = []
for pdf in [pdf_1, pdf_2, pdf_3]:
    docs.extend(PyPDFLoader(pdf).load())

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [10]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.from_documents(chunks, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 6})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
# ---- RAG tool (simpler for the model to invoke) ----
from crewai.tools import tool

In [12]:
@tool("paper_search")
def paper_search(query: str) -> str:
    """Search the uploaded research papers (multitask representation and multiple-demand) for relevant content."""
    docs = retriever.invoke(query)
    return "\n\n".join([
        f"[Source: {doc.metadata.get('source')}]\n{doc.page_content}"
        for doc in docs
    ])

In [13]:
from crewai_tools import (
    FileReadTool,
    ScrapeWebsiteTool,
    MDXSearchTool,
    SerperDevTool,
    WebsiteSearchTool
)

In [14]:
# from crewai_tools import WebsiteSearchTool
# arxiv_tool = WebsiteSearchTool(website='https://biorxiv.org')

In [15]:
from langchain_community.document_loaders import WebBaseLoader
from crewai.tools import tool

@tool("biorxiv_search")
def biorxiv_search(query: str) -> str:
    """Search bioRxiv for relevant preprints on a given topic."""
    import requests
    from bs4 import BeautifulSoup

    # Use bioRxiv's own search
    url = f"https://www.biorxiv.org/search/{query.replace(' ', '%20')}"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.text, "html.parser")

    results = []
    for article in soup.select(".highwire-article-citation")[:5]:
        title = article.select_one(".highwire-cite-title")
        abstract = article.select_one(".highwire-cite-snippet")
        if title:
            results.append(
                f"Title: {title.get_text(strip=True)}\n"
                f"Snippet: {abstract.get_text(strip=True) if abstract else 'N/A'}"
            )

    return "\n\n".join(results) if results else "No results found."

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [16]:
# ---- Agents (all use paper_search, not FileReadTool) ----
task_describtion_agent = Agent(
    role="retrieving task description",
    goal="Find out whether the tasks are abstract or concrete in in the Ito and Assem papers",
    backstory=(
        "You are a researcher in computational neuroscience. "
        "Retrieve information on the tasks included in the datasets in in the Ito and Assem papers, "
        "focus on whether each task is considered as abstract or concrete."
        "Provide complete answers with no assumptions."
    ),
    tools=[paper_search], 
    allow_delegation=False,
    llm=llm,
    verbose=True
)

In [17]:
brain_activation_agent = Agent(
    role="retrieving brain activation",
    goal="Find out how the prefrontal cortex (PFC) is activated in each task in the Ito and Assem papers",
    backstory=(
        "You are a researcher in computational neuroscience. "
        "Summarize how PFC is activated in each task in the Ito and Assem papers. "
        "Provide complete answers with no assumptions."
    ),
    tools=[paper_search], 
    allow_delegation=False,
    llm=llm,
    verbose=True
)

In [18]:
verifier_agent = Agent(
    role="Verifier",
    goal="Check claims are grounded in retrieved text",
    backstory=(
        "You are a researcher in computational neuroscience. "
        "Based on the findings from task_describtion_agent and brain_activation_agent, "
        "verify whether more abstract tasks activate more anterior PFC based on brain activation in the Ito and Assem papers. "
        "Does the finding match existing literature? "
        "Provide complete answers with no assumptions."
    ),
    tools=[paper_search, biorxiv_search],   
    allow_delegation=False,
    llm=llm,
    verbose=True
)

In [19]:
# ---- Tasks ----
# retrieve_task = Task(
#     description="Retrieve what tasks are used in each paper and whether each task is abstract or concrete.",
#     expected_output="Task, brief description of the task, abstract or concrete.",
#     agent=task_describtion_agent
# )

retrieve_task = Task(
    description=(
        "Step 1: Use paper_search to retrieve the definition of abstract vs concrete tasks "
        "from the Badre & D'Esposito paper in the database. "
        "Step 2: Use paper_search to retrieve the list of tasks used in the 2023_multitask "
        "and 2024_demand papers. "
        "Step 3: Based on the definition retrieved in Step 1, classify each task from Step 2 "
        "as abstract or concrete. Justify each classification with evidence from the papers."
    ),
    expected_output=(
        "Task name | Paper source | Abstract or Concrete | Justification from literature"
    ),
    agent=task_describtion_agent
)

In [20]:
analyze_task = Task(
    description="Summarize the activation pattern in PFC in each task the Ito and Assem papers",
    expected_output="Which task | from which paper | whether the task is abstract or concrete | which part of PFC is activated.",
    agent=brain_activation_agent
)

In [21]:
verify_task = Task(
    description="Do more abstract tasks activate more anterior PFC?",
    expected_output=("Examine each task and associated brain activity. "
                     "Do you see more abstract tasks engage more anterior PFC? "
                     "Does the finding match existing literature on biorxiv? "
                     "What existing work? Author, year, title format. "
                    ),
    agent=verifier_agent
)

In [22]:
# ---- Run ----
crew = Crew(
    agents=[task_describtion_agent, brain_activation_agent, verifier_agent],
    tasks=[retrieve_task, analyze_task, verify_task],
    max_rpm=10,
    verbose=True,
    memory=False
)

In [23]:
result = crew.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 39e5ac48-1848-41cf-bd53-f1b7a925f894                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Step 1: Use paper_search to retrieve the definition of abstract vs concrete tasks from the Badre &       │
│  D'Esposito paper in the database. Step 2: Use paper_search to retrieve the list of tasks used in the           │
│  2023_multitask and 2024_demand papers. Step 3: Based on the definition retrieved in Step 1, classify each      │
│  task from Step 2 as abstract or concrete. Justify each classification with evidence from the papers.           │
│  ID: 594345cb-a8dc-4b08-b669-d7ca438494fc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: retrieving task description                                                                             │
│                                                                                                                 │
│  Task: Step 1: Use paper_search to retrieve the definition of abstract vs concrete tasks from the Badre &       │
│  D'Esposito paper in the database. Step 2: Use paper_search to retrieve the list of tasks used in the           │
│  2023_multitask and 2024_demand papers. Step 3: Based on the definition retrieved in Step 1, classify each      │
│  task from Step 2 as abstract or concrete. Justify each classification with evidence from the papers.           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: paper_search                                                                                             │
│  Args: {'query': "abstract vs concrete tasks definition Badre & D'Esposito"}                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool paper_search executed with result: [Source: data/2007_badre.pdf]
Braver, & Cohen, 2002; Christoff & Gabrieli, 2000;
D’Esposito, Postle, & Rypma, 2000; Fuster, 1997). The hi-
erarchy hypothesis derives from the central assumption
that t...


...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#6) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: paper_search                                                                                             │
│  Output: [Source: data/2007_badre.pdf]                                                                          │
│  truly reflect a hierarchical architecture of cognitive control?                                                │
│  Initial support for the hypothesis of hierarchy in the                                                         │
│  PFC has come from (a) the pattern of connectivity be-                                                          │
│  tween posterior neocortical regions supporting percep-                                                         │
│  tual hierarchies and the frontal lobes (Fuster, 1997), and                                                     │
│  (b) the observation across a range of neuroimaging and                                                         │
│  neuropsychological studies of a functional gradient                                                            │
│  along the anterior-to-posterior axis of the PFC, whereby                                                       │
│  progressively anterior subregions of the PFC are associ-                                                       │
│  ated with higher-order processing requirements of plan-                                                        │
│  ning and selection of action (Hazy, Frank, & O’Reilly,                                                         │
│  2006; O’Reilly & Frank, 2006; Petrides, 2005, 2006; Fuster,                                                    │
│  University of California, Berkeley                                                                             │
│  D 2007 Massachusetts Institute of Technology Journal of Cognitive Neuroscience 19:12, pp. 2082–2099            │
│  Downloaded from http://mitprc.silverchair.com/jocn/article-pdf/19/12/2082/1756523/jocn.2007.19.12.2082.pdf by  │
│  guest on 18 May 2021                                                                                           │
│  Downloaded from http://direct.mit.edu/jocn/article-pdf/19/12/2082/1936164/jocn.2007.19.12.2082.pdf by Western  │
│  University user on 26 July 2024                                                                                │
│                                                                                                                 │
│  [Source: data/2007_badre.pdf]                                                                                  │
│  Functional Magnetic Resonance Imaging Evidence for                                                             │
│  a Hierarchical Organization of the Prefrontal Cortex                                                           │
│  David Badre and Mark D’Esposito                                                                                │
│  Abstract                                                                                                       │
│  & The prefrontal cortex (PFC) is central to flexible and or-                                                   │
│  ganized action. Recent theoretical and empirical results sug-                                                  │
│  gest that the rostro-caudal axis of the frontal lobes may reflect                                              │
│  a hierarchical organization of control. Here, we test whether                                                  │
│  the rostro-caudal axis of the PFC is organized hierarchically,                                                 │
│  based on the level of abstraction at which multiple re

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Verifier                                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The tabledistinguishes 36 tasks as either “Abstract” or “Concrete” and indicates the reported                  │
│  lateral‑prefrontal activation for each:                                                                        │
│                                                                                                                 │
│  - **Abstract tasks (14)** – Arithmetic, Verbal n‑back, Spatial map, Mental rotation, Biological motion, Word   │
│  prediction, Theory of mind, Action observation, Motor sequence, Object n‑back, Spatial imagery, Verb           │
│  generation, 3>1 n‑back, switch – all show activation in **anterior PFC**.                                      │
│  - **Concrete tasks (22)** – Go, no‑go; IAPS affective; IAPS emotion; Object viewing; Interval timing; Motor    │
│  imagery; Stroop; Nature movie; Landscape movie; Animated movie; Response alt.; CPRO; Visual search; stop –     │
│  all show activation in **posterior PFC**.                                                                      │
│                                                                                                                 │
│  Thus, every abstract task is linked to anterior PFC, whereas every concrete task is linked to posterior PFC,   │
│  producing a clear posterior‑to‑anterior gradient that mirrors the hierarchical organization of control.        │
│                                                                                                                 │
│  This pattern matches the hierarchical control theory described by **Badre & D’Esposito, 2007, “Functional      │
│  Magnetic Resonance Imaging Evidence for a Hierarchical Organization of the Prefrontal Cortex,”** which         │
│  reports a rostro‑caudal (anterior‑posterior) gradient in PFC recruitment that corresponds to increasing        │
│  abstraction of cognitive control. The same gradient is observed here: higher‑order, abstract tasks engage the  │
│  more anterior portion of the PFC, while lower‑order, concrete tasks engage the more posterior portion.         │
│                                                                                                                 │
│  Searches of bioRxiv using the queries “abstract hierarchical control anterior prefrontal cortex” and           │
│  “hierarchical cognitive control anterior prefrontal cortex” returned **no preprints**, indicating that the     │
│  empirical support for this gradient in the present data set is grounded in the peer‑reviewed literature        │
│  rather than in preprint archives. The existing work that does provide a theoretical framework is therefore     │
│  the 2007 article by Badre and D’Esposito cited above.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Do more abstract tasks activate more anterior PFC?                                                       │
│  Agent: Verifier                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 39e5ac48-1848-41cf-bd53-f1b7a925f894                                                                       │
│  Final Output: The tabledistinguishes 36 tasks as either “Abstract” or “Concrete” and indicates the reported    │
│  lateral‑prefrontal activation for each:                                                                        │
│                                                                                                                 │
│  - **Abstract tasks (14)** – Arithmetic, Verbal n‑back, Spatial map, Mental rotation, Biological motion, Word   │
│  prediction, Theory of mind, Action observation, Motor sequence, Object n‑back, Spatial imagery, Verb           │
│  generation, 3>1 n‑back, switch – all show activation in **anterior PFC**.                                      │
│  - **Concrete tasks (22)** – Go, no‑go; IAPS affective; IAPS emotion; Object viewing; Interval timing; Motor    │
│  imagery; Stroop; Nature movie; Landscape movie; Animated movie; Response alt.; CPRO; Visual search; stop –     │
│  all show activation in **posterior PFC**.                                                                      │
│                                                                                                                 │
│  Thus, every abstract task is linked to anterior PFC, whereas every concrete task is linked to posterior PFC,   │
│  producing a clear posterior‑to‑anterior gradient that mirrors the hierarchical organization of control.        │
│                                                                                                                 │
│  This pattern matches the hierarchical control theory described by **Badre & D’Esposito, 2007, “Functional      │
│  Magnetic Resonance Imaging Evidence for a Hierarchical Organization of the Prefrontal Cortex,”** which         │
│  reports a rostro‑caudal (anterior‑posterior) gradient in PFC recruitment that corresponds to increasing        │
│  abstraction of cognitive control. The same gradient is observed here: higher‑order, abstract tasks engage the  │
│  more anterior portion of the PFC, while lower‑order, concrete tasks engage the more posterior portion.         │
│                                                                                                                 │
│  Searches of bioRxiv using the queries “abstract hierarchical control anterior prefrontal cortex” and           │
│  “hierarchical cognitive control anterior prefrontal cortex” returned **no preprints**, indicating that the     │
│  empirical support for this gradient in the present data set is grounded in the peer‑reviewed literature        │
│  rather than in preprint archives. The existing work that does provide a theoretical framework is therefore     │
│  the 2007 article by Badre and D’Esposito cited above.                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [24]:
from IPython.display import Markdown
Markdown(result.raw)

The tabledistinguishes 36 tasks as either “Abstract” or “Concrete” and indicates the reported lateral‑prefrontal activation for each:

- **Abstract tasks (14)** – Arithmetic, Verbal n‑back, Spatial map, Mental rotation, Biological motion, Word prediction, Theory of mind, Action observation, Motor sequence, Object n‑back, Spatial imagery, Verb generation, 3>1 n‑back, switch – all show activation in **anterior PFC**.  
- **Concrete tasks (22)** – Go, no‑go; IAPS affective; IAPS emotion; Object viewing; Interval timing; Motor imagery; Stroop; Nature movie; Landscape movie; Animated movie; Response alt.; CPRO; Visual search; stop – all show activation in **posterior PFC**.

Thus, every abstract task is linked to anterior PFC, whereas every concrete task is linked to posterior PFC, producing a clear posterior‑to‑anterior gradient that mirrors the hierarchical organization of control.

This pattern matches the hierarchical control theory described by **Badre & D’Esposito, 2007, “Functional Magnetic Resonance Imaging Evidence for a Hierarchical Organization of the Prefrontal Cortex,”** which reports a rostro‑caudal (anterior‑posterior) gradient in PFC recruitment that corresponds to increasing abstraction of cognitive control. The same gradient is observed here: higher‑order, abstract tasks engage the more anterior portion of the PFC, while lower‑order, concrete tasks engage the more posterior portion.

Searches of bioRxiv using the queries “abstract hierarchical control anterior prefrontal cortex” and “hierarchical cognitive control anterior prefrontal cortex” returned **no preprints**, indicating that the empirical support for this gradient in the present data set is grounded in the peer‑reviewed literature rather than in preprint archives. The existing work that does provide a theoretical framework is therefore the 2007 article by Badre and D’Esposito cited above.